Deflection of a plate using the CLPT
---
Engineering Mechanics Stability

Graduate School Course

Author: Saullo G. P. Castro

Date: 8 July 2025



In [4]:
import numpy as np
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import eigsh
from composites import isotropic_plate

from legendre import vecf, vecfxi, vecfxixi
from legendre_gauss_quadrature import legendre_gauss_quadrature

# approximation order
m1 = m2 = 20
N = m1*m2
print('m1 = %d' % m1)
print('m2 = %d' % m2)
print('N = %d' % N)
i = np.arange(m1)
j = np.arange(m2)
pts1, weights1 = legendre_gauss_quadrature(2*m1 - 1)
pts2, weights2 = legendre_gauss_quadrature(2*m2 - 1)

# Material properties
E = 200.e9
nu = 0.3
G = E/(2*(1 + nu))

# Boundary conditions (simply supported)
xit1 = 0
xir1 = 0
xit2 = 0
xir2 = 0
etat1 = 0
etar1 = 0
etat2 = 0
etar2 = 0

# Geometric properties
a = 3
b = 7
h = 0.005

# Calculating ABD matrices
prop = isotropic_plate(thickness=h, E=E, nu=nu)

# Applied load
Pforce = 1.

buff = np.zeros((N, N))
K = np.zeros((N, N))
Fext = np.zeros((N,))

def addouter(matrix, vec1, vec2):
    np.outer(vec1, vec2, out=buff)
    matrix += buff

# stiffness matrix
# numerical integration in 2D using Legendre-Gauss quadrature
for xi, wxi in zip(pts1, weights1):
    P_xi = vecf(m1, xi, xit1, xir1, xit2, xir2)
    Px_xi = vecfxi(m1, xi, xit1, xir1, xit2, xir2)
    Pxx_xi = vecfxixi(m1, xi, xit1, xir1, xit2, xir2)
    for eta, weta in zip(pts2, weights2):
        P_eta = vecf(m2, eta, etat1, etar1, etat2, etar2)
        Px_eta = vecfxi(m2, eta, etat1, etar1, etat2, etar2)
        Pxx_eta = vecfxixi(m2, eta, etat1, etar1, etat2, etar2)

        weight = wxi*weta

        Pi, Pj = np.meshgrid(P_xi, P_eta, indexing='ij')  
        Sw = (Pi*Pj).flatten()
        
        Pxi, Pj = np.meshgrid(Px_xi, P_eta, indexing='ij')
        Swx = (Pxi*Pj*(2/a)).flatten()
        
        Pi, Pxj = np.meshgrid(P_xi, Px_eta, indexing='ij')
        Swy = (Pi*Pxj*(2/b)).flatten()

        Pxxi, Pj = np.meshgrid(Pxx_xi, P_eta, indexing='ij')
        Swxx = (Pxxi*Pj*(2/a)**2).flatten()

        Pi, Pxxj = np.meshgrid(P_xi, Pxx_eta, indexing='ij')
        Swyy = (Pi*Pxxj*(2/b)**2).flatten()

        Pxi, Pxj = np.meshgrid(Px_xi, Px_eta, indexing='ij')
        Swxy = (Pxi*Pxj*(2/a)*(2/b)).flatten()
        
        
        e1xx = -Swxx
        e1yy = -Swyy
        e1xy = -2*Swxy
        
        Mxx = prop.D11*e1xx + prop.D12*e1yy + prop.D16*e1xy
        Myy = prop.D12*e1xx + prop.D22*e1yy + prop.D26*e1xy
        Mxy = prop.D16*e1xx + prop.D26*e1yy + prop.D66*e1xy
        
        detJ = a*b/4
        
        # stiffness matrix
        addouter(K, detJ*weight*Mxx, e1xx)
        addouter(K, detJ*weight*Myy, e1yy)
        addouter(K, detJ*weight*Mxy, e1xy)
        

# external force vector
xi = 0 # x = a/2
eta = 0 # y = b/2
P_xi = vecf(m1, xi, xit1, xir1, xit2, xir2)
P_eta = vecf(m2, eta, etat1, etar1, etat2, etar2)
Pi, Pj = np.meshgrid(P_xi, P_eta, indexing='ij')  
Sw = (Pi*Pj).flatten()
Fext = Pforce*Sw


m1 = 20
m2 = 20
N = 400


In [5]:
from scipy.sparse.linalg import spsolve
from scipy.sparse import csc_matrix

#eigvals, eigvecs = eigsh(A=csc_matrix(KG), k=3, which='SM',
#                         M=csc_matrix(KNL), tol=0, sigma=1., mode='cayley')
from structsolve import solve
u = solve(csc_matrix(K), Fext)

			Removing null columns...
				144 columns removed
			finished!


In [6]:
wcentre = Sw@u
print('w at centre', wcentre)
print('w reference', 6.594931610258557e-05) # from https://github.com/saullocastro/pyfe3d/blob/main/tests/test_quad4_static_point_load.py

w at centre 2.774985496231098e-05
w reference 6.594931610258557e-05
